# Prepare the ShapeNet subset for training (CPU, no GPU needed)

The authors' preprocessed data is 645 GB (`occupancies.zip` 589 GB, `surfaces.zip` 56 GB), far too big for Kaggle. This notebook runs `kaggle/prepare_subset.py`, which:
- streams **only the chosen categories** straight from the zips on Google Drive (airplane + chair: about 133 GB downloaded, nothing stored locally),
- keeps a random **50k volume, 50k near-surface and 30k surface points per shape** (of 500k, 500k and 100k; training draws 1024 + 1024 and 2048 per sample),
- writes about **16 GB** to `/kaggle/working/shapenet_subset`, in the layout the training code expects, split files included.

**Settings** (right-hand panel): Accelerator **None** (CPU is enough). Internet **on**.

**Run it with Save Version → Save & Run All (Commit)**, so it keeps going after you close the tab. It takes roughly 1–3 hours, depending on Google Drive's speed. The last cell checks that every shape in the split files has all its files.

**Then**, in the training notebook, click **Add Input** and add this notebook's output. The training notebook finds the data on its own.

In [ ]:
REPO_URL = "https://github.com/ugwnate-hash/3DShape2VecSet.git"
BRANCH = "fix/review-issues"
CATEGORIES = "02691156,03001627"     # airplane, chair (must match the training config)
OUT = "/kaggle/working/shapenet_subset"
REPO_DIR = "/tmp/3DShape2VecSet"

import os, subprocess, sys, shutil, glob
shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)

In [ ]:
proc = subprocess.Popen([sys.executable, "-u", f"{REPO_DIR}/kaggle/prepare_subset.py", "--out", OUT,
                         "--categories", CATEGORIES], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "prepare_subset.py failed"

In [ ]:
# Every shape listed in train/val/test.lst must have its occupancy file, scale file and surface file.
ok = True
for cat in CATEGORIES.split(","):
    for split in ("train", "val", "test"):
        lst = f"{OUT}/ShapeNetV2_point/{cat}/{split}.lst"
        if not os.path.exists(lst):
            print(f"MISSING {lst}"); ok = False; continue
        ids = [l.strip().replace(".npz", "") for l in open(lst) if l.strip()]
        missing = [i for i in ids if not (os.path.exists(f"{OUT}/ShapeNetV2_point/{cat}/{i}.npz")
                                          and os.path.exists(f"{OUT}/ShapeNetV2_point/{cat}/{i}.npy")
                                          and os.path.exists(f"{OUT}/ShapeNetV2_watertight/{cat}/4_pointcloud/{i}.npz"))]
        print(f"{cat} {split:5s}: {len(ids):5d} shapes, {len(missing)} incomplete")
        ok &= not missing
size = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(OUT) for f in fs)
print(f"total size: {size / 1e9:.1f} GB")
assert ok, "some shapes are incomplete: rerun the notebook"
print("Data is complete.")